In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("heart.csv")

#Data Cleaning
# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

print("Cleaned Data:")
print(df.head())

Cleaned Data:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  


In [8]:
#Data Integration
# Split and merge (simulation)

df1 = df[['age', 'sex', 'cp']]
df2 = df[['age', 'chol', 'thalach', 'target']]

df_integrated = pd.merge(df1, df2, on='age')

print("Integrated Data:")
print(df_integrated.head())

Integrated Data:
   age  sex  cp  chol  thalach  target
0   52    1   0   212      168       0
1   52    1   0   204      156       0
2   52    1   0   201      158       1
3   52    1   0   186      190       1
4   52    1   0   223      169       1


In [9]:
#Error Correcting
# Remove invalid negative values
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].apply(lambda x: x if x >= 0 else np.nan)

# Fill again
df.fillna(df.mean(numeric_only=True), inplace=True)

print("Error Corrected Data:")
print(df.head())

Error Corrected Data:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  


In [17]:
#Data Transformation
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Restore original target column from df2 as df's target column was scaled in previous run
# This assumes df2 was correctly created from df before any erroneous scaling of target
# We check if 'target' exists in df2 and if df['target'] is currently continuous before overwriting.
if 'target' in df2.columns and not pd.api.types.is_integer_dtype(df['target']):
    df['target'] = df2['target']

scaler = StandardScaler()
# Exclude the 'target' column from numerical columns to be scaled
num_cols = df.select_dtypes(include=np.number).columns.drop('target', errors='ignore')

df[num_cols] = scaler.fit_transform(df[num_cols])

print("Transformed Data:")
print(df.head())

Transformed Data:
        age       sex        cp  trestbps      chol       fbs   restecg  \
0 -0.267966  0.682656 -0.935208 -0.376556 -0.667728 -0.418446  0.901657   
1 -0.157260  0.682656 -0.935208  0.478910 -0.841918  2.389793 -1.002541   
2  1.724733  0.682656 -0.935208  0.764066 -1.403197 -0.418446  0.901657   
3  0.728383  0.682656 -0.935208  0.935159 -0.841918 -0.418446  0.901657   
4  0.839089 -1.464866 -0.935208  0.364848  0.919336  2.389793  0.901657   

    thalach     exang   oldpeak     slope        ca      thal  target  
0  0.806035 -0.698344 -0.037124  0.979514  1.274980  1.119967       0  
1  0.237495  1.431958  1.773958 -2.271182 -0.714911  1.119967       0  
2 -1.074521  1.431958  1.342748 -2.271182 -0.714911  1.119967       0  
3  0.499898 -0.698344 -0.899544  0.979514  0.280034  1.119967       0  
4 -1.905464 -0.698344  0.739054 -0.645834  2.269926 -0.513994       0  


In [18]:
#Data Model Building
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df.drop('target', axis=1)
y = df['target'] # Use the target from df to match X's sample size

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8032786885245902
